# waveletcnn_opt on the FPGA

The optimized kernel: the same quantized WaveletCNN as `waveletcnn`, but the
whole capture in one start, all of the work in hardware. The kernel reads the raw L1a
cube straight from DDR, drops the 10 unused bands, z-scores the rest, cuts the image
into 32 x 32 patches (its last row and column repeated to fill the edge patches) and
writes every pixel's 3 scores and its class into place. The ARM only starts it. Inside,
a DATAFLOW chain: the DDR readers fill a strip ping-pong buffer in block RAM while the
fused 1D DWT + 1x1 projection, the 3x3 conv_block and the 1x1 seg_head stream every
pixel in integer fixed point. The answers are checked bit for bit against the PC's
integer model of the kernel; against the float model, the int8 input moves a few
pixels' scores, but hardly any class.

Cloud / land / sea classification of a HYPSO-2 capture: highest score = class:
**0 cloud, 1 land, 2 sea**.

Next to this notebook: the kernel's `.bit` + `.hwh` and its register map `x..._hw.h`:
`waveletcnn_opt` (ZCU104) or a board's version, such as `waveletcnn_opt_zynq7020`
(PYNQ-Z2), which reads the cube through fewer ports and has no cycle counter. The notebook finds which from the
files. And an image zip made on the PC with
`python3 tools/waveletcnn_image.py prepare ... --out aegean_waveletcnn` and `fpga_zip aegean_waveletcnn`
(the same image works for both kernels). Run the cells from the top.

The image zip can also go in the folder above this one, once for every design: the notebook looks there when it is not next to it, unpacks it there, and writes its own results next to itself.

## 1. Load the bitstream

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate, Clocks

# Which kernel: its register map, x<NAME>_hw.h, lies next to its NAME.bit.
header_file = next(Path('.').glob('x*_hw.h'))
NAME = header_file.name[1:-len('_hw.h')]
overlay = Overlay(f'{NAME}.bit')                 # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = header_file.read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, HEIGHT, WIDTH = register('AP_CTRL'), register('HEIGHT_DATA'), register('WIDTH_DATA')
DINS = [int(a, 16) for a in re.findall(r'_ADDR_DIN\d+_DATA\s+(0x[0-9a-fA-F]+)', header)]   # din0, din1, ...
DOUT = register('DOUT_DATA')
# cycles: the kernel's own count of its clock cycles, from its start until the last score
# is in memory, without the ARM's register writes and polling. A board's version without
# that register has None here.
CYCLES = register('CYCLES_DATA') if '_ADDR_CYCLES_DATA' in header else None
MHZ = Clocks.fclk0_mhz                          # the kernel's clock, as the overlay set it

def counted():
    # The last start's time in seconds, by the kernel's own count (None without the register).
    return None if CYCLES is None else kernel.read(CYCLES) / (MHZ * 1e6)

# The kernel's formats (waveletcnn_opt.hpp): in, the raw L1a cube as captured
# (H x W x 120 uint16); out, H x W x 4 int32: the 3 scores = score * 2^OUT_FRAC, then
# the class (0 cloud, 1 land, 2 sea).
BANDS, OUT_FRAC, LANES = 120, 15, 4

def launch(cube_address, scores_address, height, width):
    # Start the kernel on a height x width cube; every input port reads it.
    for offset, address in [(d, cube_address) for d in DINS] + [(DOUT, scores_address)]:
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    kernel.write(HEIGHT, height)
    kernel.write(WIDTH, width)
    kernel.write(AP_CTRL, 1)

def wait(timeout=60):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')

def to_float(scores):
    return np.multiply(np.asarray(scores), np.float32(2.0 ** -OUT_FRAC), dtype=np.float32)

print(f'{NAME}, {len(DINS)} input ports, {MHZ:.1f} MHz, '
      f'{"a" if CYCLES is not None else "no"} cycle counter, idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. Load the image into DDR

The zip holds two answers from the PC: `reference_fixed.npy`, what the kernel must
write, bit for bit (the integer model), and `reference_scores.npy`, the float model's
scores.

The image zip is on the SD card, next to this notebook; it is unpacked there (only when
the zip is newer than what is unpacked). The raw cube then goes from the SD card into
DDR that the FPGA can read (PYNQ's CMA: 512 MB, shared by every running notebook --
shut down the ones you are not using). If the whole cube does not fit, it stays in
ordinary memory and goes to the FPGA in strips of 128 rows, two strip buffers taking
turns.

In [ ]:
# The image: IMAGE.zip or the folder IMAGE/, next to this notebook or in the folder above
# it. Above is shared by every design's folder: upload the image there once. A zip is
# unpacked next to itself, and again only when it is newer than what was unpacked.
IMAGE = 'aegean_waveletcnn'
place = next((d for d in (Path('.'), Path('..'))
              if (d / f'{IMAGE}.zip').exists() or (d / IMAGE / 'cube.npy').exists()), None)
if place is None:
    raise FileNotFoundError(f'no {IMAGE}.zip or {IMAGE}/ in {Path(".").resolve()} or the folder above it')
folder, archive = place / IMAGE, place / f'{IMAGE}.zip'
if archive.exists() and (not (folder / 'cube.npy').exists()
                         or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
    with zipfile.ZipFile(archive) as z:
        z.extractall(place)
print(f'image: {folder.resolve()}')
t0 = time.monotonic()
cube = np.load(folder / 'cube.npy', mmap_mode='r')     # read from the SD card as it is used
if cube.dtype != np.uint16:                           # an image folder from before: counts as floats
    cube = np.rint(cube).astype(np.uint16)
# The PC's answers (waveletcnn_image.py prepare), if the image has them: the numpy
# model's scores, and the kernel's exact output. Without them, only the labels check it.
def from_pc(name):
    return np.load(folder / name) if (folder / name).exists() else None
reference, reference_fixed = from_pc('reference_scores.npy'), from_pc('reference_fixed.npy')
if reference_fixed is None:
    print(f'no reference_fixed.npy in {folder}: checked against the labels only')
meta = json.loads((folder / 'meta.json').read_text())
H, W, _ = cube.shape
PATCHES = -(-H // 32) * -(-W // 32)

scores_out = allocate(shape=(H, W, LANES), dtype=np.int32)
STRIP = 128                                           # rows per strip, if the cube does not fit
try:
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
    cube_in[:] = cube
    cube_in.flush()                                   # into DDR, not just the CPU cache
    strips = None
except Exception as error:                            # not enough (contiguous) CMA
    cube_in = None
    cube = np.ascontiguousarray(cube)                 # all of it in memory, for the strips
    strips = [allocate(shape=(STRIP, W, BANDS), dtype=np.uint16) for _ in range(2)]
    print(f'the whole cube does not fit in CMA ({error}): strips of {STRIP} rows instead')
print(f'{H} x {W} pixels, {PATCHES} patches, loaded in {time.monotonic() - t0:.1f} s')

## 3. Self-test: the first patch against the PC's answer

In [ ]:
# The top-left patch on its own, in its own buffers.
patch_in = allocate(shape=(min(H, 32), min(W, 32), BANDS), dtype=np.uint16)
patch_out = allocate(shape=patch_in.shape[:2] + (LANES,), dtype=np.int32)
patch_in[:] = cube[:32, :32]
patch_in.flush()
launch(patch_in.physical_address, patch_out.physical_address, *patch_in.shape[:2])
wait()
patch_out.invalidate()
if reference_fixed is not None:
    wrong = int((np.asarray(patch_out) != reference_fixed[:32, :32]).sum())
    print('PASS' if wrong == 0 else 'FAIL', f'{wrong} values differ from the PC integer model')
else:
    print('no reference: the first patch ran, its answers are not checked')

## 4. The whole image

**Wall clock**: from calling `classify` until the ARM can read the scores and classes
(int32: score * 2^OUT_FRAC, and the class the kernel picked). **FPGA**: from the start bit
until the done bit, as the ARM sees it. **FPGA, by its own count**: the kernel's
cycle counter, from its start until the last score is in memory: the kernel's time alone. The scores are turned into floats only for the checks below,
untimed: the classes are the answer.

In [ ]:
def classify(timeout=60):
    # The loaded cube -> (H x W x 4 int32: scores and class, seconds the FPGA ran, as the
    # ARM saw it and by the kernel's own count; None without the counter).
    if strips is None:
        t = time.perf_counter()
        launch(cube_in.physical_address, scores_out.physical_address, H, W)
        wait(timeout)
        fpga = time.perf_counter() - t
        own = counted()
    else:
        # Double-buffered strips: the ARM copies strip s + 1 while the FPGA does strip s.
        def fill(s):
            rows = cube[s * STRIP:(s + 1) * STRIP]
            strips[s % 2][:len(rows)] = rows
            strips[s % 2].flush()
            return len(rows)
        count, fpga, own = -(-H // STRIP), 0.0, None if CYCLES is None else 0.0
        rows = fill(0)
        for s in range(count):
            t = time.perf_counter()
            launch(strips[s % 2].physical_address,
                   scores_out.physical_address + s * STRIP * W * LANES * 4, rows, W)
            next_rows = fill(s + 1) if s + 1 < count else 0
            wait(timeout)
            fpga += time.perf_counter() - t           # with the copy, if that took longer
            if own is not None:
                own += counted()                      # the kernel alone, without the copy
            rows = next_rows
    scores_out.invalidate()                           # read what the kernel wrote
    return np.asarray(scores_out), fpga, own

t = time.perf_counter()
raw_scores, fpga, own = classify()
wall = time.perf_counter() - t
classes = raw_scores[..., 3]
print(f'wall clock: {1000 * wall:7.1f} ms for {H} x {W} pixels, {PATCHES} patches')
print(f'FPGA:       {1000 * fpga:7.1f} ms, {1e6 * fpga / PATCHES:.1f} us per patch'
      + ('' if strips is None else f' (in strips of {STRIP} rows, with any copy that outlasted the FPGA)'))
if own is not None:
    print(f'FPGA, by its own count: {1000 * own:7.3f} ms, {1e6 * own / PATCHES:.1f} us per patch, '
          f'{own * MHZ * 1e6 / PATCHES:.0f} cycles per patch at {MHZ:.0f} MHz')

## 5. Check it: against the labels, and the PC's answer if the image has it

In [ ]:
scores = to_float(raw_scores)[..., :3]    # all 4 lanes in one pass, then drop the class
np.save(f'{IMAGE}_fpga_scores.npy', scores)    # next to this notebook: the image folder is shared
close = None if reference_fixed is None else raw_scores == reference_fixed
if close is not None:
    print(f'bit-exact against the PC integer model: {close.all(-1).mean():.2%} of pixels')
if reference is not None:
    print(f'same class as the PC float model: {(classes == reference.argmax(-1)).mean():.3%}, '
          f'scores within 0.1 of it: {(np.abs(scores - reference) <= 0.1).all(-1).mean():.2%}')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = cube[:, :, meta['rgb_bands']].astype(np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          'FPGA classes': COLORS[classes]}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
    wrong = np.full(labels.shape + (3,), 255, dtype=np.uint8)   # white: same class as the label
    wrong[~known] = 128                                         # gray: unlabeled
    wrong[known & (classes != labels)] = (220, 30, 30)          # red: a different class
    panels['differs from labels'] = wrong
elif close is not None:
    panels['differs from PC'] = ~close.all(-1)
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # classes: cloud white, land green, sea blue, gray = unlabeled

## 6. Speed: the FPGA alone

The kernel again on the cube already in DDR (in strips: on one strip), best of 5, as the
ARM sees it and by the kernel's own count.

In [ ]:
buffer, rows = (cube_in, H) if strips is None else (strips[0], min(STRIP, H))
if strips is not None:
    strips[0][:rows] = cube[:rows]
    strips[0].flush()
runs, own_runs = [], []
for _ in range(5):
    t = time.perf_counter()
    launch(buffer.physical_address, scores_out.physical_address, rows, W)
    wait()
    runs.append(time.perf_counter() - t)
    own_runs.append(counted())
n = -(-rows // 32) * -(-W // 32)
best = min(runs)
print(f'FPGA alone: {1000 * best:.1f} ms for {rows} x {W} pixels ({n} patches), {1e6 * best / n:.1f} us per patch')
if CYCLES is not None:
    own_best = min(own_runs)
    print(f'by its own count: {1000 * own_best:.3f} ms, {1e6 * own_best / n:.1f} us per patch, '
          f'{own_best * MHZ * 1e6 / n:.0f} cycles per patch at {MHZ:.0f} MHz')
print(f'the whole image: about {1000 * best * PATCHES / n:.1f} ms on the FPGA, {1000 * wall:.1f} ms wall clock')